In [1]:
"""SESSION 35: YOLO12n-Seg five-fold CV on SkinDisNet_2_Annotation.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import multiprocessing
import random
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


SESSION_NAME = "SESSION_35"
MODEL_CONFIG = "yolo12n-seg.yaml"
INITIAL_WEIGHTS = "yolo12n.pt"
FOLD_COUNT = 5
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_ALIASES = {
    "train": ("train",),
    "val": ("valid", "val", "validation"),
    "test": ("test",),
}


def _ensure_dependencies() -> None:
    print("[install] Installing/upgrading Ultralytics for YOLO12-Seg support.", flush=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "ultralytics"
    ])
    import ultralytics

    print(
        f"[environment] ultralytics={getattr(ultralytics, '__version__', 'unknown')}",
        flush=True,
    )


def _has_split(root: Path, split: str) -> bool:
    return any(
        (root / name / "images").is_dir() and (root / name / "labels").is_dir()
        for name in SPLIT_ALIASES[split]
    )


def _valid_root(root: Path) -> bool:
    return (root / "data.yaml").is_file() and all(
        _has_split(root, split) for split in SPLIT_ALIASES
    )


def _has_source_manifest(root: Path) -> bool:
    path = root / "manifest.csv"
    if not path.is_file():
        return False
    with path.open(newline="", encoding="utf-8") as stream:
        columns = set(next(csv.reader(stream), []))
    return {"name", "split", "source", "augmented"} <= columns


def _resolve_root(hint: str) -> Path:
    if hint != "auto":
        root = Path(hint).expanduser().resolve()
        if _valid_root(root):
            return root
        if root.is_dir():
            candidates = sorted({
                path.parent.resolve()
                for path in root.rglob("data.yaml")
                if _valid_root(path.parent)
            })
            preferred = [
                path for path in candidates
                if _has_source_manifest(path)
                or path.name.lower().replace("-", "_") == "skin_disease_yolo"
            ]
            if len(preferred) == 1:
                return preferred[0]
            if len(candidates) == 1:
                return candidates[0]
        raise FileNotFoundError(f"Incomplete YOLO dataset at {root}.")

    candidates: set[Path] = set()
    for search_root in (Path("/kaggle/input"), Path.cwd()):
        if search_root.is_dir():
            for yaml_file in search_root.rglob("data.yaml"):
                if _valid_root(yaml_file.parent):
                    candidates.add(yaml_file.parent.resolve())
    manifested = sorted(path for path in candidates if _has_source_manifest(path))
    preferred = manifested or sorted(
        path for path in candidates
        if path.name.lower().replace("-", "_") == "skin_disease_yolo"
    )
    selected = preferred or sorted(candidates)
    if not selected:
        raise FileNotFoundError(
            "Could not find a supported YOLO dataset. Attach "
            "SkinDisNet_2_Annotation in Kaggle or pass --data-root "
            "/kaggle/input/<dataset-slug>."
        )
    if len(selected) != 1:
        raise RuntimeError(
            "Multiple datasets found; attach one SkinDisNet_2_Annotation dataset "
            "or pass --data-root explicitly:\n"
            + "\n".join(f"  {path}" for path in selected)
        )
    return selected[0]


def _read_classes_and_splits(root: Path) -> tuple[list[str], dict[str, str]]:
    try:
        import yaml
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "PyYAML"])
        import yaml

    config = yaml.safe_load((root / "data.yaml").read_text(encoding="utf-8")) or {}
    raw_names = config.get("names")
    if isinstance(raw_names, dict):
        names = [str(raw_names[key]) for key in sorted(raw_names, key=lambda key: int(key))]
    elif isinstance(raw_names, list):
        names = [str(name) for name in raw_names]
    else:
        raise ValueError("data.yaml must contain class names as a list or numeric mapping.")
    if not names:
        raise ValueError("No classes were declared in data.yaml.")
    if config.get("nc") is not None and int(config["nc"]) != len(names):
        raise ValueError(f"data.yaml nc={config['nc']} conflicts with {len(names)} class names.")

    splits = {}
    for split, aliases in SPLIT_ALIASES.items():
        folder = next(
            (name for name in aliases if (root / name / "images").is_dir()), None
        )
        if folder is None:
            raise FileNotFoundError(f"Missing image directory for split '{split}'.")
        splits[split] = folder
    return names, splits


def _images(root: Path, folder: str) -> list[Path]:
    return sorted(
        path for path in (root / folder / "images").iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )


def _validate_dataset(
    root: Path, names: list[str], splits: dict[str, str]
) -> dict[str, int]:
    counts = {}
    for split, folder in splits.items():
        images = _images(root, folder)
        if not images:
            raise ValueError(f"No images found for split '{split}'.")
        for image in images:
            label = root / folder / "labels" / f"{image.stem}.txt"
            if not label.is_file():
                raise FileNotFoundError(f"Missing label file: {label}")
            for line_number, line in enumerate(label.read_text(encoding="utf-8").splitlines(), 1):
                values = line.split()
                if not values:
                    continue
                if len(values) < 7 or len(values) % 2 == 0:
                    raise ValueError(f"Invalid polygon row in {label}:{line_number}")
                try:
                    class_number = float(values[0])
                    coordinates = [float(value) for value in values[1:]]
                except ValueError as error:
                    raise ValueError(f"Non-numeric polygon row in {label}:{line_number}") from error
                class_id = int(class_number)
                if class_number != class_id or not 0 <= class_id < len(names):
                    raise ValueError(f"Invalid class id in {label}:{line_number}")
                if any(not 0 <= value <= 1 for value in coordinates):
                    raise ValueError(f"Non-normalized polygon coordinates in {label}:{line_number}")
        counts[split] = len(images)
    return counts


def _collect_cv_items(root: Path, splits: dict[str, str]) -> list[dict[str, Any]]:
    source_by_image: dict[tuple[str, str], str] = {}
    manifest_path = root / "manifest.csv"
    if _has_source_manifest(root):
        with manifest_path.open(newline="", encoding="utf-8") as stream:
            reader = csv.DictReader(stream)
            for row in reader:
                split = row.get("split", "").strip().lower()
                split = "val" if split in {"valid", "validation"} else split
                name = Path(row.get("name", "")).stem
                source = row.get("source", "").strip()
                key = (split, name)
                if (
                    split not in SPLIT_ALIASES or not name or not source
                    or key in source_by_image
                ):
                    raise ValueError(
                        f"Invalid or duplicate source manifest row in {manifest_path}: {row}"
                    )
                source_by_image[key] = source
        if not source_by_image:
            raise ValueError(f"Source manifest contains no image rows: {manifest_path}")

    items = []
    found_manifest_images: set[tuple[str, str]] = set()
    source_splits: dict[str, set[str]] = {}
    for split in ("train", "val", "test"):
        folder = splits[split]
        label_dir = root / folder / "labels"
        for image in _images(root, folder):
            key = (split, image.stem)
            source = source_by_image.get(key)
            if source_by_image and source is None:
                raise ValueError(
                    f"Image {image} is missing from source manifest {manifest_path}."
                )
            if source_by_image:
                found_manifest_images.add(key)
                source_splits.setdefault(source, set()).add(split)
            if split == "test":
                continue
            label = label_dir / f"{image.stem}.txt"
            classes = sorted({
                int(line.split()[0])
                for line in label.read_text(encoding="utf-8").splitlines()
                if line.strip()
            })
            items.append({
                "image": str(image.resolve()),
                "group": source or image.stem.split(".rf.", 1)[0],
                "classes": classes,
                "manifest_source": bool(source_by_image),
            })
    if source_by_image:
        missing = set(source_by_image) - found_manifest_images
        if missing:
            raise ValueError(
                f"{len(missing)} source manifest entries have no matching image; "
                f"first missing entry: {sorted(missing)[0]}"
            )
        overlaps = [
            source for source, source_splits_for_image in source_splits.items()
            if "test" in source_splits_for_image and len(source_splits_for_image) > 1
        ]
        if overlaps:
            raise ValueError(
                "Source manifest overlaps the locked test split; first source: "
                + sorted(overlaps)[0]
            )
    return items


def _assign_folds(
    items: list[dict[str, Any]], class_count: int, seed: int
) -> tuple[list[list[int]], dict[str, Any]]:
    groups: dict[str, list[int]] = {}
    class_totals = [0] * class_count
    for index, item in enumerate(items):
        groups.setdefault(item["group"], []).append(index)
        for class_id in item["classes"]:
            class_totals[class_id] += 1
    if len(groups) < FOLD_COUNT:
        raise ValueError(f"Need at least {FOLD_COUNT} unique source groups.")

    grouped = []
    for group_name, indices in groups.items():
        class_counts = [0] * class_count
        for index in indices:
            for class_id in items[index]["classes"]:
                class_counts[class_id] += 1
        rarity = sum(count / max(total, 1) for count, total in zip(class_counts, class_totals))
        grouped.append((group_name, indices, class_counts, rarity))
    rng = random.Random(seed)
    tie_break = {group[0]: rng.random() for group in grouped}
    grouped.sort(key=lambda item: (-item[3], -len(item[1]), tie_break[item[0]], item[0]))

    target_size = len(items) / FOLD_COUNT
    target_classes = [total / FOLD_COUNT for total in class_totals]
    fold_indices: list[list[int]] = [[] for _ in range(FOLD_COUNT)]
    fold_counts = [[0] * class_count for _ in range(FOLD_COUNT)]
    for _, indices, class_counts, _ in grouped:
        costs = []
        for fold_id in range(FOLD_COUNT):
            size_cost = ((len(fold_indices[fold_id]) + len(indices)) / target_size) ** 2
            class_cost = sum(
                ((fold_counts[fold_id][class_id] + count)
                 / max(target_classes[class_id], 1)) ** 2
                for class_id, count in enumerate(class_counts)
            )
            costs.append((size_cost + class_cost, len(fold_indices[fold_id]), fold_id))
        chosen = min(costs)[2]
        fold_indices[chosen].extend(indices)
        for class_id, count in enumerate(class_counts):
            fold_counts[chosen][class_id] += count
    if any(not indices for indices in fold_indices):
        raise RuntimeError("CV assignment created an empty fold.")

    report = {
        "fold_count": FOLD_COUNT,
        "seed": seed,
        "image_count": len(items),
        "source_group_count": len(groups),
        "fold_validation_images": [len(indices) for indices in fold_indices],
        "fold_training_images": [len(items) - len(indices) for indices in fold_indices],
        "fold_class_image_counts": fold_counts,
        "class_image_totals": class_totals,
        "grouping_rule": (
            "manifest source lineage; source variants stay in one fold"
            if any(item.get("manifest_source") for item in items)
            else "all filename variants before '.rf.' are assigned together"
        ),
    }
    return fold_indices, report


def _write_fold_configs(
    work_dir: Path,
    items: list[dict[str, Any]],
    fold_indices: list[list[int]],
    names: list[str],
    seed: int,
) -> list[Path]:
    import yaml

    all_indices = set(range(len(items)))
    output = []
    for fold_id, held_out in enumerate(fold_indices):
        held_out_set = set(held_out)
        train_indices = sorted(all_indices - held_out_set)
        valid_indices = sorted(held_out_set)
        valid_groups = {items[index]["group"] for index in valid_indices}
        if any(items[index]["group"] in valid_groups for index in train_indices):
            raise RuntimeError(f"Source-group leakage detected in fold {fold_id}.")
        fold_dir = work_dir / "folds" / f"seed_{seed}" / f"fold_{fold_id}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        train_list = fold_dir / "train.txt"
        valid_list = fold_dir / "valid.txt"
        train_list.write_text(
            "\n".join(items[index]["image"] for index in train_indices) + "\n",
            encoding="utf-8",
        )
        valid_list.write_text(
            "\n".join(items[index]["image"] for index in valid_indices) + "\n",
            encoding="utf-8",
        )
        yaml_path = fold_dir / "data.yaml"
        yaml_path.write_text(
            yaml.safe_dump(
                {
                    "train": str(train_list.resolve()),
                    "val": str(valid_list.resolve()),
                    "nc": len(names),
                    "names": names,
                },
                sort_keys=False,
                allow_unicode=True,
            ),
            encoding="utf-8",
        )
        output.append(yaml_path)
    return output


def _metric_value(owner: Any, attribute: str) -> float | None:
    value = getattr(owner, attribute, None)
    try:
        result = float(value)
    except (TypeError, ValueError):
        return None
    return result if result == result else None


def _metrics(metrics: Any, names: list[str]) -> dict[str, Any]:
    segment = getattr(metrics, "seg", None)
    if segment is None:
        raise RuntimeError("YOLO12-Seg validation returned no segmentation metrics.")
    per_class: dict[str, float | None] = {name: None for name in names}
    ap = getattr(segment, "ap", None)
    class_indices = getattr(segment, "ap_class_index", None)
    if ap is not None and class_indices is not None:
        class_indices = (
            class_indices.tolist()
            if hasattr(class_indices, "tolist") else list(class_indices)
        )
        ap_values = ap.tolist() if hasattr(ap, "tolist") else list(ap)
        for class_id, class_ap in zip(class_indices, ap_values):
            if not 0 <= int(class_id) < len(names):
                continue
            if isinstance(class_ap, (list, tuple)):
                if not class_ap:
                    continue
                class_ap = class_ap[0]
            try:
                score = float(class_ap)
            except (TypeError, ValueError):
                continue
            if score == score:
                per_class[names[int(class_id)]] = score
    box = getattr(metrics, "box", None)
    return {
        "segm_mAP50": _metric_value(segment, "map50"),
        "segm_mAP50_95": _metric_value(segment, "map"),
        "per_class_segm_mAP50": per_class,
        "bbox_mAP50": _metric_value(box, "map50"),
        "bbox_mAP50_95": _metric_value(box, "map"),
    }


def _write_class_csv(path: Path, values: dict[str, float | None]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=["class", "segm_mAP50"])
        writer.writeheader()
        writer.writerows(
            {"class": name, "segm_mAP50": score} for name, score in values.items()
        )


def _completed_epochs(path: Path) -> int:
    if not path.is_file():
        return 0
    with path.open(newline="", encoding="utf-8") as stream:
        return sum(1 for _ in csv.DictReader(stream))


def _aggregate(results: list[dict[str, Any]], names: list[str]) -> dict[str, Any]:
    output: dict[str, Any] = {}
    for metric in ("segm_mAP50", "segm_mAP50_95", "bbox_mAP50", "bbox_mAP50_95"):
        values = [
            float(item["metrics"][metric]) for item in results
            if item["metrics"].get(metric) is not None
        ]
        output[metric] = {
            "mean": statistics.fmean(values) if values else None,
            "std": statistics.pstdev(values) if len(values) > 1 else 0.0 if values else None,
            "fold_count": len(values),
        }
    per_class = {}
    for name in names:
        values = [
            float(item["metrics"]["per_class_segm_mAP50"][name])
            for item in results
            if item["metrics"]["per_class_segm_mAP50"].get(name) is not None
        ]
        per_class[name] = {
            "mean": statistics.fmean(values) if values else None,
            "std": statistics.pstdev(values) if len(values) > 1 else 0.0 if values else None,
            "fold_count": len(values),
        }
    output["per_class_segm_mAP50"] = per_class
    return output


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="SESSION 35: YOLO12n-Seg group-safe five-fold CV"
    )
    parser.add_argument(
        "--data-root", default="auto",
        help="Dataset directory; auto-detects SkinDisNet_2_Annotation on Kaggle.",
    )
    parser.add_argument("--work-dir", default=None)
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--time-budget-hours", type=float, default=10.5)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--workers", type=int, default=8)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--patience", type=int, default=20)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--init-weights", default=INITIAL_WEIGHTS,
        help="Detection weights for compatible-layer transfer; use 'none' for scratch.",
    )
    parser.add_argument("--resume", action="store_true")
    args, _ = parser.parse_known_args(argv)
    started = time.monotonic()
    if (args.epochs < 1 or args.imgsz < 32 or args.batch < 1
            or args.workers < 0 or args.time_budget_hours <= 0):
        raise ValueError("Epochs, image size, batch, and time budget must be positive.")

    _ensure_dependencies()
    from ultralytics import YOLO

    root = _resolve_root(args.data_root)
    names, split_dirs = _read_classes_and_splits(root)
    image_counts = _validate_dataset(root, names, split_dirs)
    cv_items = _collect_cv_items(root, split_dirs)
    folds, split_report = _assign_folds(cv_items, len(names), args.seed)
    source_group_count = split_report["source_group_count"]
    dataset_summary_path = root / "summary.json"
    dataset_evaluation_note = None
    if dataset_summary_path.is_file():
        dataset_summary = json.loads(
            dataset_summary_path.read_text(encoding="utf-8")
        )
        dataset_evaluation_note = dataset_summary.get("evaluation_note")

    device = args.device
    if device == "auto":
        import torch

        device = ",".join(str(i) for i in range(torch.cuda.device_count())) \
            if torch.cuda.is_available() else "cpu"
    work_dir = (
        Path(args.work_dir).expanduser().resolve()
        if args.work_dir
        else Path("/kaggle/working/session_35_yolo12n_seg_skindisnet2")
        if Path("/kaggle/working").is_dir()
        else Path.cwd() / "session_35_yolo12n_seg_skindisnet2"
    )
    work_dir.mkdir(parents=True, exist_ok=True)
    fold_yamls = _write_fold_configs(work_dir, cv_items, folds, names, args.seed)
    split_signature = hashlib.sha256(json.dumps(
        {"items": cv_items, "folds": folds, "names": names, "seed": args.seed},
        sort_keys=True,
    ).encode("utf-8")).hexdigest()
    split_report_path = work_dir / "cv_split_report.json"
    if args.resume and split_report_path.is_file():
        previous = json.loads(split_report_path.read_text(encoding="utf-8"))
        if previous.get("split_signature") != split_signature:
            raise RuntimeError(
                "Dataset or CV split differs from the saved run. Restore the original "
                "dataset and seed, or use a new --work-dir."
            )
    split_report["split_signature"] = split_signature
    split_report_path.write_text(
        json.dumps(split_report, indent=2), encoding="utf-8"
    )

    print("[model] YOLO12n-Seg has no official pretrained segmentation checkpoint.", flush=True)
    if args.init_weights.lower() == "none":
        print("[model] training the full segmentation architecture from scratch.", flush=True)
    else:
        print(
            f"[model] transferring compatible detection weights from {args.init_weights}; "
            "new segmentation head will be trained.",
            flush=True,
        )
    print(
        f"[data] {root}; train+valid pooled={len(cv_items)}; "
        f"held-out external test={image_counts['test']}; classes={len(names)}; "
        f"development source groups={source_group_count}",
        flush=True,
    )
    if dataset_evaluation_note:
        print(f"[evaluation note] {dataset_evaluation_note}", flush=True)
    print(
        f"[cv] validation counts={split_report['fold_validation_images']}; "
        f"training counts={split_report['fold_training_images']}; device={device}",
        flush=True,
    )
    print(
        f"[budget] up to {args.time_budget_hours:.2f}h total; reserving 45 minutes "
        "for validation/reporting; maximum five independent fold trainings.",
        flush=True,
    )

    completed: list[dict[str, Any]] = []
    pending = []
    for fold_id, data_yaml in enumerate(fold_yamls):
        run_name = f"seed_{args.seed}_fold_{fold_id}"
        run_dir = work_dir / "runs" / run_name
        checkpoint = run_dir / "weights" / "best.pt"
        metrics_path = work_dir / "metrics" / f"{run_name}.json"
        signature_path = work_dir / "metrics" / f"{run_name}.signature"
        signature = hashlib.sha256(json.dumps({
            "split": split_signature, "model": MODEL_CONFIG, "initial_weights": args.init_weights,
            "epochs": args.epochs, "imgsz": args.imgsz, "batch": args.batch,
            "patience": args.patience, "seed": args.seed, "fold": fold_id,
        }, sort_keys=True).encode("utf-8")).hexdigest()
        if signature_path.is_file():
            previous_signature = signature_path.read_text(encoding="utf-8").strip()
            if previous_signature != signature:
                raise RuntimeError(
                    f"Fold {fold_id} training settings differ from the saved run; "
                    "choose a new --work-dir."
                )
        elif (args.resume and (run_dir / "weights" / "last.pt").is_file()
              and not metrics_path.is_file()):
            raise RuntimeError(
                f"Cannot safely resume fold {fold_id}: its settings signature is "
                f"missing at {signature_path}. Use a new --work-dir."
            )
        else:
            signature_path.parent.mkdir(parents=True, exist_ok=True)
            signature_path.write_text(signature + "\n", encoding="utf-8")
        if metrics_path.is_file():
            result = json.loads(metrics_path.read_text(encoding="utf-8"))
            if result.get("fold_signature") != signature:
                raise RuntimeError(
                    f"Existing fold {fold_id} used different settings; set a new --work-dir."
                )
            if not checkpoint.is_file():
                raise FileNotFoundError(f"Missing completed fold checkpoint: {checkpoint}")
            completed.append(result)
        else:
            pending.append((fold_id, data_yaml, signature))

    for pending_index, (fold_id, data_yaml, signature) in enumerate(pending):
        remaining_folds = len(pending) - pending_index
        remaining_hours = args.time_budget_hours - (time.monotonic() - started) / 3600
        training_hours = (remaining_hours - 0.75) / remaining_folds
        if training_hours <= 0.05:
            raise TimeoutError("Insufficient run budget remains for every pending fold.")
        run_name = f"seed_{args.seed}_fold_{fold_id}"
        run_dir = work_dir / "runs" / run_name
        last_checkpoint = run_dir / "weights" / "last.pt"
        best_checkpoint = run_dir / "weights" / "best.pt"
        resume = args.resume and last_checkpoint.is_file()
        if last_checkpoint.is_file() and not resume:
            raise RuntimeError(
                f"Fold {fold_id} has an incomplete checkpoint. Rerun with --resume "
                "or select a new --work-dir."
            )
        if resume:
            model = YOLO(str(last_checkpoint))
        else:
            model = YOLO(MODEL_CONFIG)
            if args.init_weights.lower() != "none":
                model.load(args.init_weights)

        fold_started = time.monotonic()
        print(
            f"[fold {fold_id}] training {split_report['fold_training_images'][fold_id]} "
            f"images; validation={split_report['fold_validation_images'][fold_id]}; "
            f"time limit={training_hours:.2f}h; resume={resume}",
            flush=True,
        )
        model.train(
            data=str(data_yaml),
            task="segment",
            epochs=args.epochs,
            time=training_hours,
            imgsz=args.imgsz,
            batch=args.batch,
            workers=args.workers,
            device=device,
            patience=args.patience,
            seed=args.seed + fold_id,
            project=str(work_dir / "runs"),
            name=run_name,
            exist_ok=True,
            resume=resume,
            amp=True,
            plots=True,
            save=True,
            verbose=True,
        )
        del model
        if not best_checkpoint.is_file():
            raise FileNotFoundError(
                f"Fold {fold_id} ended without a best checkpoint: {best_checkpoint}"
            )

        evaluator = YOLO(str(best_checkpoint))
        eval_device = device.split(",")[0] if device not in ("", "cpu") else "cpu"
        metrics = evaluator.val(
            data=str(data_yaml),
            split="val",
            imgsz=args.imgsz,
            batch=min(args.batch, 8),
            workers=args.workers,
            device=eval_device,
            plots=True,
            project=str(work_dir / "runs"),
            name=f"{run_name}_evaluation",
            exist_ok=True,
            verbose=True,
        )
        result = {
            "fold": fold_id,
            "fold_signature": signature,
            "metrics": _metrics(metrics, names),
            "best_checkpoint": str(best_checkpoint),
            "epochs_completed": _completed_epochs(run_dir / "results.csv"),
            "elapsed_seconds": time.monotonic() - fold_started,
        }
        metrics_path = work_dir / "metrics" / f"{run_name}.json"
        metrics_path.parent.mkdir(parents=True, exist_ok=True)
        metrics_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
        _write_class_csv(
            work_dir / "metrics" / f"{run_name}_per_class_segm_map50.csv",
            result["metrics"]["per_class_segm_mAP50"],
        )
        completed.append(result)
        print(
            f"[fold {fold_id}] segmentation mAP50="
            f"{result['metrics']['segm_mAP50']}; epochs={result['epochs_completed']}",
            flush=True,
        )

    completed.sort(key=lambda result: result["fold"])
    summary = {
        "session": SESSION_NAME,
        "model": "YOLO12n-Seg",
        "model_config": MODEL_CONFIG,
        "initial_detection_weights": (
            None if args.init_weights.lower() == "none" else args.init_weights
        ),
        "task": "instance segmentation",
        "dataset_root": str(root),
        "dataset_evaluation_note": dataset_evaluation_note,
        "source_grouped_cross_validation": any(
            item.get("manifest_source") for item in cv_items
        ),
        "development_source_group_count": source_group_count,
        "class_names": names,
        "image_counts": image_counts,
        "cross_validation": {
            "folds": FOLD_COUNT,
            "fold_seed": args.seed,
            "training_splits": "train + valid pooled; test excluded from CV",
            "split_report": split_report,
            "fold_results": completed,
            "aggregate_metrics": _aggregate(completed, names),
        },
        "training_plan": {
            "epoch_argument_per_fold": args.epochs,
            "time_budget_hours": args.time_budget_hours,
            "device": device,
            "image_size": args.imgsz,
            "batch": args.batch,
            "epochs_completed": {str(item["fold"]): item["epochs_completed"] for item in completed},
            "elapsed_seconds": time.monotonic() - started,
        },
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    summary_path = work_dir / "SESSION_35_results.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    aggregate = summary["cross_validation"]["aggregate_metrics"]
    _write_class_csv(
        work_dir / "cv_per_class_segm_map50.csv",
        {name: values["mean"] for name, values in aggregate["per_class_segm_mAP50"].items()},
    )
    print(
        f"[CV complete] segm mAP50={aggregate['segm_mAP50']['mean']} "
        f"+/- {aggregate['segm_mAP50']['std']}",
        flush=True,
    )
    print(f"[complete] results={summary_path}", flush=True)
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    raise SystemExit(main())


[install] Installing/upgrading Ultralytics for YOLO12-Seg support.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[environment] ultralytics=8.4.175
[model] YOLO12n-Seg has no official pretrained segmentation checkpoint.
[model] transferring compatible detection weights from yolo12n.pt; new segmentation head will be trained.
[data] /kaggle/input/datasets/munnamuntasirul/skindisnet-2-annotation; train+valid pooled=5000; held-out external test=1000; classes=6; development source groups=1319
[evaluation note] Validati

/usr/local/lib/python3.13/dist-packages/ultralytics/utils/loss.py:640: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /root/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at /pytorch/aten/src/ATen/native/cuda/jit_utils.cpp:1505.)
  marea = xyxy2xywh(target_bboxes_normalized)[..., 2:].prod(2)



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/100         2G     0.5884      2.926      3.114      1.077          0         23        640: 100% ━━━━━━━━━━━━ 250/250 3.1it/s 1:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 2.3it/s 14.0s
                   all       1004       1389      0.148      0.436      0.197      0.161      0.143      0.426      0.191      0.151


/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/69      2.36G      0.559      1.492      1.759      1.035          0         16        640: 100% ━━━━━━━━━━━━ 250/250 4.7it/s 52.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.8it/s 6.7s
                   all       1004       1389       0.64      0.326      0.193      0.155      0.635      0.315      0.184      0.142

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/86      2.36G      0.568      1.459      1.698      1.044          0         21        640: 100% ━━━━━━━━━━━━ 250/250 4.8it/s 52.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.7it/s 6.8s
                   all       1004       1

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


... (5 outputs left out)

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       2/62      2.36G     0.5436      1.468      1.757      1.033          0         23        640: 100% ━━━━━━━━━━━━ 250/250 3.9it/s 1:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.9it/s 8.1s
                   all       1000       1429      0.517      0.392      0.205       0.15      0.513      0.386      0.202      0.151

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/75      2.36G     0.5528      1.365      1.734      1.034          0         20        640: 100% ━━━━━━━━━━━━ 250/250 4.2it/s 1:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.0it/s 8.1s
                   all       1000       142

SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
